# Simulating an AI Agent for Employee Onboarding

**Copyright (c) 2026 Shrikara Kaudambady. All rights reserved.**

This notebook demonstrates how an AI Agent can answer new employee questions. The agent uses a **Retrieval-Augmented Generation (RAG)** pattern. It first retrieves relevant information from a set of internal documents and then uses that context to generate a helpful, factually-grounded answer. The final LLM call is simulated to focus on the agent's architecture.

### 1. Setup and Library Imports

In [ ]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Download necessary NLTK data (only needs to be done once)
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    print("Downloading NLTK data...")
    nltk.download('punkt')
    nltk.download('stopwords')
    print("Download complete.")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

### 2. Create the Knowledge Base

Here we define our internal company documents. In a real-world scenario, this text would be loaded from files (`.txt`, `.md`, `.pdf`, etc.).

In [ ]:
hr_policy = """
HR Policy Document 1.0
Work Hours: Standard work hours are 9:00 AM to 5:00 PM, Monday to Friday. 
Vacation Policy: Full-time employees are entitled to 20 paid vacation days per year. 
Sick Leave: Employees receive 10 days of paid sick leave annually.
"""

tech_setup = """
Technical Setup Guide for Software Engineers
To set up your local development environment, you must first install Docker Desktop. 
Once Docker is running, clone the 'main-app' repository from our GitHub. 
In the repository's root directory, run the command 'docker-compose up' to build and start all necessary services.
"""

company_values = """
Our Company Mission and Values
Our mission is to innovate relentlessly and build products that empower our customers. 
We value collaboration, transparency, and a strong commitment to quality.
"""

documents = {
    'hr_policy': hr_policy,
    'tech_setup': tech_setup,
    'company_values': company_values
}

### 3. Define and Simulate the AI Agent

We'll create an `OnboardingAgent` class. This class encapsulates the entire RAG process: retrieving documents and generating an answer.

In [ ]:
class OnboardingAgent:
    def __init__(self, documents):
        self.documents = documents
        self.doc_names = list(documents.keys())
        self.doc_contents = list(documents.values())
        self.stop_words = set(stopwords.words('english'))
        
        # Initialize the TF-IDF Vectorizer
        self.vectorizer = TfidfVectorizer(tokenizer=self._preprocess_text)
        self.doc_vectors = self.vectorizer.fit_transform(self.doc_contents)
        print("AI Agent initialized. Knowledge base has been vectorized.")
        
    def _preprocess_text(self, text):
        tokens = word_tokenize(text.lower())
        return [token for token in tokens if token.isalpha() and token not in self.stop_words]

    def _retrieve_context(self, query):
        print(f"\n-> Agent searching knowledge base for: '{query}'")
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.doc_vectors).flatten()
        
        # Get the most relevant document
        most_relevant_idx = np.argmax(similarities)
        relevance_score = similarities[most_relevant_idx]
        
        if relevance_score < 0.1: # Confidence threshold
            print("<- Agent found no relevant context.")
            return None
        
        relevant_doc_name = self.doc_names[most_relevant_idx]
        relevant_doc_content = self.doc_contents[most_relevant_idx]
        print(f"<- Agent found most relevant context in '{relevant_doc_name}' with score {relevance_score:.2f}")
        return relevant_doc_content

    def _simulate_llm_call(self, query, context):
        print("-> Agent is constructing prompt for LLM.")
        if context is None:
            prompt = f"Question: {query}\n\nAnswer:"
            # Simulated LLM response when there's no context
            simulated_response = "I'm sorry, but I couldn't find any information about that in my knowledge base. Please ask a question related to HR policies, tech setup, or company values."
        else:
            prompt = f"Use the following context to answer the question.\n\nContext: {context}\n\nQuestion: {query}\n\nAnswer:"
            # Simulated LLM response when context is found
            simulated_response = f"Based on the document I found, here is the answer to your question about '{query}': ... (simulated LLM summary of context would go here) ..."
        
        print("<- Agent generated the final answer.")
        return simulated_response, prompt
        
    def get_answer(self, query):
        print("=========================================================")
        context = self._retrieve_context(query)
        answer, prompt = self._simulate_llm_call(query, context)
        print("\n--- LLM PROMPT (for demonstration) ---")
        print(prompt)
        print("---------------------------------------")
        print("\n*** AGENT'S FINAL ANSWER ***")
        print(answer)
        print("=========================================================\n")


### 4. Demonstrate the Agent in Action

Let's instantiate our agent and ask it some typical new employee questions.

In [ ]:
agent = OnboardingAgent(documents)

# Question 1: A clear question from the HR policy
agent.get_answer("How much vacation time do I get per year?")

# Question 2: A technical question
agent.get_answer("How do I set up my dev environment?")

# Question 3: A question about company culture
agent.get_answer("What is the company's mission?")

# Question 4: A question the agent cannot answer
agent.get_answer("What time does the cafeteria close?")